# Expériences de révision — MPRF (IJDMMM-342531)

Deux cellules indépendantes, à lancer dans Colab avec `1. risk_factors_cervical_cancer.csv` dans `/content` (même fichier que le papier).

- **Cellule A** — règles évaluées sur les **folds de test** (le Tableau 6 du papier est calculé sur la validation) + fidélité des règles à la forêt élaguée. Répond à R1.9 et R2.2.
- **Cellule B** — (1) imputation médiane **dans** les folds (le papier la fait avant le CV), (2) retrait des variables Hinselmann / Schiller / Citology / Dx*. Répond à R1.2, R1.3 et R2.1. Le run 1 doit reproduire le Tableau 2 : sinon, arrêter et me montrer la sortie.

Colle-moi les sorties complètes, je les intègre au manuscrit.

In [ ]:
# ============================================================
# MPRF — Règles évaluées sur les FOLDS DE TEST (réponse R1.9 / R2.2)
# Dataset : Risk Factors Cervical Cancer
# Mêmes folds, mêmes arbres, mêmes seuils que le papier ; ajoute les métriques TEST + fidélité à la forêt élaguée
# Google Colab — copier-coller directement
# ============================================================

import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score
import warnings
warnings.filterwarnings("ignore")

# ── Chargement ───────────────────────────────────────────────
df = pd.read_csv("/content/1. risk_factors_cervical_cancer.csv")
df.replace("?", np.nan, inplace=True)
df = df.apply(pd.to_numeric, errors="coerce")
df.fillna(df.median(numeric_only=True), inplace=True)

TARGET = "Biopsy"
feature_cols = [c for c in df.columns if c != TARGET]
X = df[feature_cols].values
y = df[TARGET].values.astype(int)
feature_names = feature_cols

print(f"Shape : {X.shape}  |  Classes : {np.bincount(y)}  |  "
      f"IR ≈ {np.bincount(y)[0]/np.bincount(y)[1]:.1f}:1")

# ── Paramètres ───────────────────────────────────────────────
K_BIRF      = 10
ALPHA       = 0.60
N_TREES     = 50
N_SPLITS    = 10
MIN_SUPPORT = 0.05   # ← MODIFIER ICI — seuil support Stage 1
MIN_PREC    = 0.30   # ← MODIFIER ICI — seuil précision Stage 1

# ════════════════════════════════════════════════════════════
# FONCTIONS MPRF
# ════════════════════════════════════════════════════════════

def ensemble_predict(trees, X, indices):
    preds = np.array([trees[i].predict(X) for i in indices])
    return (preds.sum(axis=0) >= len(indices) / 2).astype(int)

def combined_score(y_true, y_pred, alpha):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    gm = np.sqrt(recall_1 * recall_0)
    return alpha * f1 + (1 - alpha) * gm

def MPRF(trees, X_val, y_val, k, fold, alpha=ALPHA):
    n = len(trees)
    rng_birf = np.random.RandomState(100 + fold)
    init = rng_birf.randint(0, n)
    selected  = [init]
    remaining = list(range(n))
    remaining.remove(init)
    best_score = combined_score(y_val,
                                ensemble_predict(trees, X_val, selected), alpha)
    while len(selected) < k and remaining:
        best_cand, best_cand_score = None, -np.inf
        for idx in remaining:
            score = combined_score(y_val,
                                   ensemble_predict(trees, X_val, selected + [idx]),
                                   alpha)
            if score > best_cand_score:
                best_cand_score, best_cand = score, idx
        if best_cand_score >= best_score:
            selected.append(best_cand)
            remaining.remove(best_cand)
            if best_cand_score > best_score:
                best_score = best_cand_score
        else:
            break
    return selected

# ════════════════════════════════════════════════════════════
# EXTRACTION DE RÈGLES
# ════════════════════════════════════════════════════════════

def extract_rules_from_tree(tree, feature_names, class_label=1):
    tree_  = tree.tree_
    feat_n = [feature_names[i] if i != -2 else "undefined"
              for i in tree_.feature]
    rules  = []

    def recurse(node, conditions):
        if tree_.feature[node] != -2:
            name  = feat_n[node]
            thresh = tree_.threshold[node]
            recurse(tree_.children_left[node],
                    conditions + [(name, "<=", thresh)])
            recurse(tree_.children_right[node],
                    conditions + [(name, ">", thresh)])
        else:
            if np.argmax(tree_.value[node]) == class_label:
                rules.append(list(conditions))

    recurse(0, [])
    return rules

def extract_rules_from_forest(trees, selected_indices, feature_names):
    all_rules = []
    for idx in selected_indices:
        all_rules.extend(
            extract_rules_from_tree(trees[idx], feature_names)
        )
    return all_rules

def rule_covers(rule, X):
    mask = np.ones(len(X), dtype=bool)
    for (feat, op, thresh) in rule:
        feat_idx = feature_names.index(feat)
        if op == "<=":
            mask &= (X[:, feat_idx] <= thresh)
        else:
            mask &= (X[:, feat_idx] > thresh)
    return mask

def rule_stats(rule, X, y):
    mask             = rule_covers(rule, X)
    covered          = np.sum(mask)
    if covered == 0:
        return 0.0, 0.0, 0
    minority_covered = np.sum(y[mask] == 1)
    total_minority   = np.sum(y == 1)
    support   = minority_covered / total_minority if total_minority > 0 else 0.0
    precision = minority_covered / covered
    return support, precision, covered

# ════════════════════════════════════════════════════════════
# MÉTRIQUES D'ÉVALUATION DE LA BASE DE RÈGLES
# ════════════════════════════════════════════════════════════

def evaluate_ruleset(rules, X, y, stage_name=""):
    """
    Évalue une base de règles sur (X, y) avec les métriques suivantes :
    - Nombre de règles
    - Conditions moyennes par règle
    - Couverture globale : fraction des minoritaires couverts par ≥1 règle
    - Précision moyenne : moyenne des précisions individuelles
    - Lift moyen : précision_règle / taux_base_minoritaire
    - Overlap moyen : nombre moyen de règles couvrant chaque instance minoritaire
    - Complexité : nb_règles × conditions_moyennes
    """
    if len(rules) == 0:
        return {
            "stage"         : stage_name,
            "n_rules"       : 0,
            "avg_conditions": 0.0,
            "coverage"      : 0.0,
            "avg_precision" : 0.0,
            "avg_lift"      : 0.0,
            "avg_overlap"   : 0.0,
            "complexity"    : 0.0,
        }

    base_rate    = np.sum(y == 1) / len(y)
    total_min    = np.sum(y == 1)
    minority_idx = np.where(y == 1)[0]

    # Précision, lift, support par règle
    precisions = []
    supports   = []
    lifts      = []
    n_conds    = []

    for rule in rules:
        sup, prec, cov = rule_stats(rule, X, y)
        precisions.append(prec)
        supports.append(sup)
        lifts.append(prec / base_rate if base_rate > 0 else 0.0)
        n_conds.append(len(rule))

    # Couverture globale : union des instances minoritaires couvertes
    covered_minority = np.zeros(len(X), dtype=bool)
    for rule in rules:
        covered_minority |= rule_covers(rule, X)
    coverage = np.sum(covered_minority[minority_idx]) / total_min \
               if total_min > 0 else 0.0

    # Overlap : pour chaque instance minoritaire, combien de règles la couvrent
    overlap_counts = np.zeros(total_min)
    for rule in rules:
        mask = rule_covers(rule, X)
        for i, idx in enumerate(minority_idx):
            if mask[idx]:
                overlap_counts[i] += 1
    avg_overlap = np.mean(overlap_counts[overlap_counts > 0]) \
                  if np.any(overlap_counts > 0) else 0.0

    avg_conditions = np.mean(n_conds)
    complexity     = len(rules) * avg_conditions

    return {
        "stage"         : stage_name,
        "n_rules"       : len(rules),
        "avg_conditions": round(avg_conditions, 2),
        "coverage"      : round(coverage, 3),
        "avg_precision" : round(np.mean(precisions), 3),
        "avg_lift"      : round(np.mean(lifts), 3),
        "avg_overlap"   : round(avg_overlap, 2),
        "complexity"    : round(complexity, 1),
    }

def print_metrics_table(metrics_list):
    """Affiche le tableau de comparaison des métriques par stage."""
    header = (f"  {'Étape':<30} {'Règles':>7} {'Cond/R':>7} "
              f"{'Couv.':>7} {'Préc.':>7} {'Lift':>7} "
              f"{'Overlap':>8} {'Compl.':>8}")
    print(header)
    print(f"  {'-'*76}")
    for m in metrics_list:
        print(f"  {m['stage']:<30} {m['n_rules']:>7} {m['avg_conditions']:>7.1f} "
              f"{m['coverage']:>7.3f} {m['avg_precision']:>7.3f} "
              f"{m['avg_lift']:>7.2f} {m['avg_overlap']:>8.2f} "
              f"{m['complexity']:>8.1f}")

# ════════════════════════════════════════════════════════════
# STAGES DE SIMPLIFICATION
# ════════════════════════════════════════════════════════════

def stage1_selection(rules, X_val, y_val,
                     min_support=MIN_SUPPORT, min_prec=MIN_PREC):
    scored = []
    for rule in rules:
        support, precision, covered = rule_stats(rule, X_val, y_val)
        if support >= min_support and precision >= min_prec:
            scored.append((rule, support, precision))
    scored.sort(key=lambda x: x[2], reverse=True)
    return [r[0] for r in scored], scored

def stage2_condition_pruning(rules, X_val, y_val):
    pruned_rules = []
    for rule in rules:
        _, original_prec, _ = rule_stats(rule, X_val, y_val)
        current_rule = list(rule)
        changed = True
        while changed:
            changed = False
            for i in range(len(current_rule)):
                candidate = current_rule[:i] + current_rule[i+1:]
                if len(candidate) == 0:
                    break
                _, cand_prec, cand_covered = rule_stats(candidate, X_val, y_val)
                if cand_covered > 0 and cand_prec >= original_prec:
                    current_rule  = candidate
                    original_prec = cand_prec
                    changed = True
                    break
        pruned_rules.append(current_rule)
    # Dédupliquer
    unique, seen = [], set()
    for rule in pruned_rules:
        key = tuple(sorted([(f, o, round(t, 4)) for (f, o, t) in rule]))
        if key not in seen:
            seen.add(key)
            unique.append(rule)
    return unique

def stage3_merging(rules):
    def rule_signature(rule):
        return tuple(sorted([(feat, op) for (feat, op, _) in rule]))

    groups = defaultdict(list)
    for rule in rules:
        groups[rule_signature(rule)].append(rule)

    merged = []
    for sig, group in groups.items():
        if len(group) == 1:
            merged.append(group[0])
        else:
            merged_rule = []
            for (feat, op) in sig:
                thresholds = [t for r in group
                              for (f, o, t) in r if f == feat and o == op]
                thresh = max(thresholds) if op == "<=" else min(thresholds)
                merged_rule.append((feat, op, thresh))
            merged.append(merged_rule)
    return merged

def rules_to_str(rules):
    result = []
    for i, rule in enumerate(rules):
        conds = " AND ".join(
            [f"{feat} {op} {thresh:.3f}" for (feat, op, thresh) in rule]
        )
        result.append(f"  Rule {i+1:>2} : IF {conds} THEN Biopsy=1")
    return "\n".join(result)

# ════════════════════════════════════════════════════════════
# ÉVALUATION SUR LES FOLDS DE TEST + FIDÉLITÉ À LA FORÊT ÉLAGUÉE
# ════════════════════════════════════════════════════════════

def rule_set_predict(rules, X):
    """Une instance est prédite positive si AU MOINS UNE règle la couvre."""
    pred = np.zeros(len(X), dtype=bool)
    for r in rules:
        pred |= rule_covers(r, X)
    return pred.astype(int)

def evaluate_ruleset_test(rules, X_te, y_te, forest_pred):
    n_min = int(np.sum(y_te == 1))
    base  = n_min / len(y_te)
    if len(rules) == 0:
        return dict(empty_share=np.nan, rule_prec=np.nan, coverage=0.0, set_prec=np.nan,
                    set_lift=np.nan, f1=0.0, agree=float(np.mean(forest_pred == 0)),
                    rec_vs_forest=0.0, tp=0, fp=0, n_min=n_min)
    precs, n_empty = [], 0
    for r in rules:
        _, p, cov = rule_stats(r, X_te, y_te)
        if cov == 0:
            n_empty += 1
        else:
            precs.append(p)
    pred = rule_set_predict(rules, X_te)
    tp = int(np.sum((pred == 1) & (y_te == 1)))
    fp = int(np.sum((pred == 1) & (y_te == 0)))
    set_prec = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    n_fpos = int(np.sum(forest_pred == 1))
    return dict(
        empty_share   = n_empty / len(rules),
        rule_prec     = float(np.mean(precs)) if precs else np.nan,
        coverage      = tp / n_min if n_min > 0 else np.nan,
        set_prec      = set_prec,
        set_lift      = set_prec / base if (base > 0 and not np.isnan(set_prec)) else np.nan,
        f1            = f1_score(y_te, pred, pos_label=1, zero_division=0),
        agree         = float(np.mean(pred == forest_pred)),
        rec_vs_forest = float(np.sum((pred == 1) & (forest_pred == 1)) / n_fpos) if n_fpos > 0 else np.nan,
        tp=tp, fp=fp, n_min=n_min)

def run_pipeline_test(X, y, feature_names):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    stages = ["raw", "s1", "s2", "s3"]
    test_acc = {s: defaultdict(list) for s in stages}
    val_acc  = {s: defaultdict(list) for s in stages}
    forest_f1_test, csrf_f1_test, n_min_test, n_min_val = [], [], [], []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        val_size      = int(len(train_idx) * 0.2)
        rng_split     = np.random.RandomState(fold)
        val_idx       = rng_split.choice(len(y_train), val_size, replace=False)
        train_sub_idx = np.setdiff1d(np.arange(len(y_train)), val_idx)
        X_val, y_val             = X_train[val_idx], y_train[val_idx]
        X_train_sub, y_train_sub = X_train[train_sub_idx], y_train[train_sub_idx]

        rf = RandomForestClassifier(n_estimators=N_TREES, class_weight='balanced',
                                    random_state=42, n_jobs=-1)
        rf.fit(X_train_sub, y_train_sub)
        trees = rf.estimators_

        selected    = MPRF(trees, X_val, y_val, K_BIRF, fold, ALPHA)
        forest_pred = ensemble_predict(trees, X_test, selected)

        raw_rules   = extract_rules_from_forest(trees, selected, feature_names)
        s1_rules, _ = stage1_selection(raw_rules, X_val, y_val)
        s2_rules    = stage2_condition_pruning(s1_rules, X_val, y_val)
        s3_rules    = stage3_merging(s2_rules)

        for name, rules in zip(stages, [raw_rules, s1_rules, s2_rules, s3_rules]):
            for k_, v in evaluate_ruleset_test(rules, X_test, y_test, forest_pred).items():
                test_acc[name][k_].append(v)
            m = evaluate_ruleset(rules, X_val, y_val, name)
            val_acc[name]["coverage"].append(m["coverage"])
            val_acc[name]["avg_precision"].append(m["avg_precision"])
            val_acc[name]["avg_lift"].append(m["avg_lift"])
            val_acc[name]["n_rules"].append(m["n_rules"])

        forest_f1_test.append(f1_score(y_test, forest_pred, pos_label=1, zero_division=0))
        csrf_f1_test.append(f1_score(y_test, rf.predict(X_test), pos_label=1, zero_division=0))
        n_min_test.append(int(np.sum(y_test == 1)))
        n_min_val.append(int(np.sum(y_val == 1)))
        print(f"  Fold {fold+1:>2} | arbres={len(selected):>2} | règles S3={len(s3_rules):>3} | "
              f"minoritaires test={n_min_test[-1]:>2} val={n_min_val[-1]:>2} | "
              f"couv. test S3={test_acc['s3']['coverage'][-1]:.3f} | "
              f"prec. ens. test S3={test_acc['s3']['set_prec'][-1]:.3f}")

    nm = lambda a: float(np.nanmean(a))
    print(f"\n{'='*104}")
    print("  RÈGLES : VALIDATION (in-sample, = Tableau 6 du papier)  vs  TEST (hors échantillon)")
    print(f"{'='*104}")
    print(f"  {'Étape':<8}{'Règles':>7} | {'Couv.val':>9}{'Préc.val':>9}{'Lift val':>9} | "
          f"{'Couv.test':>10}{'Préc.règle':>11}{'Préc.ens.':>10}{'Lift ens.':>10}")
    print(f"  {'-'*100}")
    for s in stages:
        print(f"  {s:<8}{nm(val_acc[s]['n_rules']):>7.1f} | {nm(val_acc[s]['coverage']):>9.3f}"
              f"{nm(val_acc[s]['avg_precision']):>9.3f}{nm(val_acc[s]['avg_lift']):>9.2f} | "
              f"{nm(test_acc[s]['coverage']):>10.3f}{nm(test_acc[s]['rule_prec']):>11.3f}"
              f"{nm(test_acc[s]['set_prec']):>10.3f}{nm(test_acc[s]['set_lift']):>10.2f}")
    print("  (Préc.règle = moyenne sur les règles couvrant >=1 instance de test ; Préc.ens. = précision de l'ensemble de règles, OR des règles)")

    print(f"\n{'='*104}")
    print("  FIDÉLITÉ À LA FORÊT ÉLAGUÉE (prédictions MPRF) — folds de test")
    print(f"{'='*104}")
    print(f"  {'Étape':<8}{'% règles sans couverture test':>31}{'F1_min règles':>15}{'Accord forêt':>14}{'Rappel vs forêt':>17}")
    for s in stages:
        print(f"  {s:<8}{nm(test_acc[s]['empty_share'])*100:>30.1f}%{nm(test_acc[s]['f1']):>15.3f}"
              f"{nm(test_acc[s]['agree']):>14.3f}{nm(test_acc[s]['rec_vs_forest']):>17.3f}")
    print(f"  Référence — F1_min test : forêt élaguée MPRF = {np.mean(forest_f1_test):.3f} | CSRF 50 arbres = {np.mean(csrf_f1_test):.3f}")

    tp = sum(test_acc['s3']['tp']); fp = sum(test_acc['s3']['fp']); nmn = sum(test_acc['s3']['n_min'])
    print(f"\n  AGRÉGÉ sur les 10 folds de test, Stage 3 : {tp}/{nmn} minoritaires couverts "
          f"(couverture {tp/nmn:.3f}), {fp} faux positifs, précision {tp/(tp+fp) if tp+fp else float('nan'):.3f}")
    print(f"  Minoritaires par fold — test : {n_min_test} | validation : {n_min_val}")
    return test_acc, val_acc

test_acc, val_acc = run_pipeline_test(X, y, feature_names)


Shape : (858, 35)  |  Classes : [803  55]  |  IR ≈ 14.6:1
  Fold  1 | arbres=10 | règles S3= 31 | minoritaires test= 5 val=11 | couv. test S3=1.000 | prec. ens. test S3=0.333
  Fold  2 | arbres= 2 | règles S3=  9 | minoritaires test= 5 val= 9 | couv. test S3=0.800 | prec. ens. test S3=0.667
  Fold  3 | arbres= 2 | règles S3=  8 | minoritaires test= 5 val=20 | couv. test S3=0.600 | prec. ens. test S3=0.500
  Fold  4 | arbres= 4 | règles S3= 16 | minoritaires test= 6 val=14 | couv. test S3=1.000 | prec. ens. test S3=0.500
  Fold  5 | arbres=10 | règles S3= 15 | minoritaires test= 6 val= 4 | couv. test S3=0.833 | prec. ens. test S3=0.455
  Fold  6 | arbres= 5 | règles S3= 22 | minoritaires test= 6 val=10 | couv. test S3=0.833 | prec. ens. test S3=0.625
  Fold  7 | arbres=10 | règles S3= 30 | minoritaires test= 6 val= 9 | couv. test S3=1.000 | prec. ens. test S3=0.667
  Fold  8 | arbres= 4 | règles S3= 11 | minoritaires test= 6 val= 9 | couv. test S3=1.000 | prec. ens. test S3=0.667
  Fold

In [ ]:
# ============================================================
# MPRF — Cervical Cancer : (1) imputation DANS les folds, (2) sans variables diagnostiques
# Réponse R1.2 / R1.3 / R2.1. Même protocole que le papier (mêmes folds, seeds, k=10, alpha=0.6).
# Google Colab — copier-coller directement
# ============================================================
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, precision_score, recall_score, average_precision_score)
import warnings
warnings.filterwarnings("ignore")

K_BIRF = 10
ALPHA  = 0.6

df = pd.read_csv("1. risk_factors_cervical_cancer.csv")
df.replace("?", np.nan, inplace=True)
df = df.apply(pd.to_numeric, errors="coerce")      # PAS d'imputation ici
TARGET = "Biopsy"
y_all = df[TARGET].values.astype(int)
print("Classes :", np.bincount(y_all), "| colonnes :", df.shape[1] - 1)

# ── Métriques ────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_proba=None):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))

    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    gmean    = np.sqrt(recall_1 * recall_0)

    metrics = {
        "f1_min"   : f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1_macro" : f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall"   : recall_1,
        "gmean"    : gmean,
    }
    if y_proba is not None:
        metrics["auprc"] = average_precision_score(y_true, y_proba)
    else:
        metrics["auprc"] = np.nan
    return metrics

# ── Vote majoritaire ─────────────────────────────────────────
def ensemble_predict(trees, X, indices):
    preds = np.array([trees[i].predict(X) for i in indices])
    return (preds.sum(axis=0) >= len(indices) / 2).astype(int)

def ensemble_proba(trees, X, indices):
    probas = np.array([trees[i].predict_proba(X)[:, 1] for i in indices])
    return probas.mean(axis=0)

# ── Score combiné pour la sélection ──────────────────────────
def combined_score(y_true, y_pred, alpha):
    """Score(S) = α·F1_min(S) + (1-α)·G-mean(S)"""
    tn = np.sum((y_true == 0) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))

    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    f1  = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    gm  = np.sqrt(recall_1 * recall_0)

    return alpha * f1 + (1 - alpha) * gm

# ── MPRF — Forward Selection hill climbing ───────────────────
def MPRF(trees, X_val, y_val, k, fold, alpha=ALPHA):
    """
    Hill climbing forward selection :
    1. Init : un arbre tiré aléatoirement (seed propre au fold)
    2. Critère : Score(S) = α·F1_min(S) + (1-α)·G-mean(S)
    3. À chaque étape : on évalue TOUS les candidats restants
       et on ajoute le meilleur même si égalité (plateau)
    4. best_score mis à jour UNIQUEMENT si amélioration stricte
    5. On s'arrête si tous les candidats dégradent ou si k atteint
    """
    n = len(trees)

    rng_birf = np.random.RandomState(100 + fold)
    init = rng_birf.randint(0, n)

    selected  = [init]
    remaining = list(range(n))
    remaining.remove(init)

    best_score = combined_score(y_val,
                                ensemble_predict(trees, X_val, selected),
                                alpha)

    while len(selected) < k and remaining:

        best_cand       = None
        best_cand_score = -np.inf

        for idx in remaining:
            score = combined_score(y_val,
                                   ensemble_predict(trees, X_val, selected + [idx]),
                                   alpha)
            if score > best_cand_score:
                best_cand_score = score
                best_cand       = idx

        # Avancer si meilleur OU égal (plateau accepté)
        # best_score mis à jour UNIQUEMENT si amélioration stricte
        if best_cand_score >= best_score:
            selected.append(best_cand)
            remaining.remove(best_cand)
            if best_cand_score > best_score:
                best_score = best_cand_score
        else:
            break

    return selected

# ── OBP ──────────────────────────────────────────────────────
def OBP(trees, X_val, y_val, max_size):
    scores = [f1_score(y_val, trees[i].predict(X_val), pos_label=1, zero_division=0)
              for i in range(len(trees))]
    order = np.argsort(scores)[::-1]
    selected, best_score = [], -np.inf
    for idx in order[:max_size]:
        cand = selected + [idx]
        score = f1_score(y_val, ensemble_predict(trees, X_val, cand),
                         pos_label=1, zero_division=0)
        if score >= best_score:
            selected.append(idx)
            best_score = score
    return selected if selected else [order[0]]

# ── IBEP ─────────────────────────────────────────────────────
def IBEP(trees, X_val, y_val, max_size):
    n = len(trees)
    preds   = np.array([trees[i].predict(X_val) for i in range(n)])
    errors  = (preds != y_val).astype(float)
    weights = np.ones(len(y_val)) / len(y_val)
    selected = []
    for _ in range(max_size):
        w_err = errors.dot(weights)
        best  = int(np.argmin(w_err))
        if best in selected:
            break
        selected.append(best)
        weights *= np.exp(errors[best])
        weights /= weights.sum()
    return selected if selected else [0]


def run_cv(df, drop_cols, impute_in_fold, n_trees=50, k=K_BIRF, n_splits=10):
    feats = [c for c in df.columns if c != TARGET and c not in drop_cols]
    X = df[feats].values.astype(float)
    y = y_all
    if not impute_in_fold:                          # protocole du papier : médiane sur tout le dataset
        med_all = np.nanmedian(X, axis=0); med_all = np.where(np.isnan(med_all), 0, med_all)
        X_all = np.where(np.isnan(X), med_all, X)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    res = defaultdict(list)
    for fold, (tr, te) in enumerate(skf.split(X, y)):
        if impute_in_fold:                          # médiane calculée sur la partie entraînement uniquement
            med = np.nanmedian(X[tr], axis=0); med = np.where(np.isnan(med), 0, med)
            X_train = np.where(np.isnan(X[tr]), med, X[tr]); X_test = np.where(np.isnan(X[te]), med, X[te])
        else:
            X_train, X_test = X_all[tr], X_all[te]
        y_train, y_test = y[tr], y[te]
        val_size      = int(len(tr) * 0.2)
        rng_split     = np.random.RandomState(fold)
        val_idx       = rng_split.choice(len(y_train), val_size, replace=False)
        train_sub_idx = np.setdiff1d(np.arange(len(y_train)), val_idx)
        X_val, y_val = X_train[val_idx], y_train[val_idx]
        X_sub, y_sub = X_train[train_sub_idx], y_train[train_sub_idx]
        rf = RandomForestClassifier(n_estimators=n_trees, class_weight='balanced', random_state=42, n_jobs=-1)
        rf.fit(X_sub, y_sub)
        trees = rf.estimators_
        outs = {
            "CSRF": (rf.predict(X_test), rf.predict_proba(X_test)[:, 1], n_trees),
        }
        for name, sel in [("MPRF", MPRF(trees, X_val, y_val, k, fold, ALPHA)),
                          ("OBP",  OBP(trees, X_val, y_val, n_trees)),
                          ("IBEP", IBEP(trees, X_val, y_val, n_trees))]:
            outs[name] = (ensemble_predict(trees, X_test, sel), ensemble_proba(trees, X_test, sel), len(sel))
        for name, (yp, ypr, size) in outs.items():
            for k_m, v in compute_metrics(y_test, yp, ypr).items():
                res[f"{name}_{k_m}"].append(v)
            res[f"{name}_size"].append(size)
    return res, len(feats)

def show(res, title, nfeat):
    print(f"\n{'='*88}\n  {title}  [{nfeat} variables]\n{'='*88}")
    for m in ["CSRF", "MPRF", "OBP", "IBEP"]:
        g = lambda key: np.mean(res[f"{m}_{key}"])
        print(f"  {m:<5} | taille={g('size'):5.1f} | F1_min={g('f1_min'):.3f} | G-mean={g('gmean'):.3f} | "
              f"AUPRC={g('auprc'):.3f} | Prec={g('precision'):.3f} | Recall={g('recall'):.3f}")

EXAMS = ["Hinselmann", "Schiller", "Citology"]
DX    = ["Dx:Cancer", "Dx:CIN", "Dx:HPV", "Dx"]

r1, n1 = run_cv(df, [],            impute_in_fold=False)
show(r1, "1) Protocole du papier (médiane globale) — doit retrouver le Tableau 2 (MPRF F1=0.661, CSRF F1=0.528)", n1)
r2, n2 = run_cv(df, [],            impute_in_fold=True)
show(r2, "2) Imputation médiane DANS les folds, toutes variables", n2)
r3, n3 = run_cv(df, EXAMS,         impute_in_fold=True)
show(r3, "3) Imputation dans les folds, SANS Hinselmann / Schiller / Citology", n3)
r4, n4 = run_cv(df, EXAMS + DX,    impute_in_fold=True)
show(r4, "4) Imputation dans les folds, SANS Hinselmann / Schiller / Citology / Dx*", n4)
print("\nF1_min MPRF par fold (4) :", np.round(r4['MPRF_f1_min'], 3))


Classes : [803  55] | colonnes : 35

  1) Protocole du papier (médiane globale) — doit retrouver le Tableau 2 (MPRF F1=0.661, CSRF F1=0.528)  [35 variables]
  CSRF  | taille= 50.0 | F1_min=0.528 | G-mean=0.656 | AUPRC=0.647 | Prec=0.652 | Recall=0.460
  MPRF  | taille=  5.5 | F1_min=0.661 | G-mean=0.803 | AUPRC=0.618 | Prec=0.662 | Recall=0.673
  OBP   | taille=  8.9 | F1_min=0.634 | G-mean=0.799 | AUPRC=0.601 | Prec=0.626 | Recall=0.667
  IBEP  | taille=  4.2 | F1_min=0.629 | G-mean=0.750 | AUPRC=0.577 | Prec=0.711 | Recall=0.583

  2) Imputation médiane DANS les folds, toutes variables  [35 variables]
  CSRF  | taille= 50.0 | F1_min=0.498 | G-mean=0.603 | AUPRC=0.671 | Prec=0.627 | Recall=0.423
  MPRF  | taille=  5.1 | F1_min=0.640 | G-mean=0.794 | AUPRC=0.619 | Prec=0.637 | Recall=0.657
  OBP   | taille=  6.7 | F1_min=0.629 | G-mean=0.792 | AUPRC=0.583 | Prec=0.635 | Recall=0.667
  IBEP  | taille=  3.6 | F1_min=0.570 | G-mean=0.741 | AUPRC=0.520 | Prec=0.603 | Recall=0.587

  3) Imp